<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>ResNet-18 전이학습(Transfer Learning with ResNet-18)</strong>에 대해 학습합니다.</br></br>
>생성 데이터로 ResNet-18을 학습하고, Knowledge Distillation 개념을 학습해봅시다.

</br>

# ResNet-18 전이학습
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성된 이미지 데이터</mark>로 ResNet-18을 전이학습하여 커스텀 분류기를 구축합니다.

ResNet-50은 약 2,500만 개의 파라미터를 갖지만, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ResNet-18은  약 1,100만 개</mark>로 절반 이하입니다. 파라미터 수가 적으면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 속도가 빠르고 GPU 메모리 사용량이 낮아</mark> 소규모 프로젝트와 실험에 적합합니다. ImageNet으로 사전학습된 ResNet-18은 이미 에지, 텍스처, 형태 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">범용적인 시각 특징</mark>을 학습한 상태이므 로, 분류 헤드만 교체하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">소량의 데이터만으로도 높은 성능</mark>을 달성할 수 있어 데이터 수집 비용을 크게 절감합니다.</br>이 장을 이해하기 위해서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ResNet 구조</mark>(잔차 블록과 Skip Connection으로 구성된 깊은 CNN), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전이학습</mark>(사전학습 모델의 가중치를 초기값으로 재활용하는 방법), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Fine-tuning</mark>(전체 또는 일부 층을 새 태스크로 재학습), 그리고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Feature Extraction</mark>(백본을 동결하고 추출된 특성만 사용하는 방식)에 대한 배경 지식이 필요합니다.

</br>

In [ ]:
# TODO 0: 필요한 라이브러리를 임포트하고, 학습에 사용할 장치를 설정해봅시다.

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
import os
from PIL import Image
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

</br>

## 합성 데이터 생성 (Synthetic Data Generation)
> 실제 데이터 수집이 어렵거나 비용이 높을 때, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">생성 모델을 활용하여 합성 데이터(Synthetic Data)를 만들어 학습에 활용</mark>할 수 있습니다.</br></br>
> 이번 실습에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Stable Diffusion</mark> 파이프라인으로 5개 클래스(cat, dog, car, airplane, flower)의 이미지를 생성하고, 이를 학습/테스트 데이터셋으로 구성합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">폴더 구조</th>
      <th style="text-align:center">설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>data/train/{클래스명}/</code></td><td style="text-align:center">학습용 이미지 (클래스별 2장)</td></tr>
    <tr><td style="text-align:center"><code>data/test/{클래스명}/</code></td><td style="text-align:center">테스트용 이미지 (클래스별 1장)</td></tr>
  </tbody>
</table>

> `ImageFolder`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">폴더 이름을 클래스 라벨로 자동 인식</mark>합니다. 위 구조로 저장하면 별도의 라벨 파일 없이도 데이터셋을 구성할 수 있습니다.

In [ ]:
# TODO 1: StableDiffusionPipeline을 로드하고, 5개 클래스의 합성 이미지를 생성하여 폴더에 저장해봅시다.

from diffusers import StableDiffusionPipeline

# TODO 1-1: StableDiffusionPipeline을 로드해봅시다. (모델: "runwayml/stable-diffusion-v1-5", dtype: float16)
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to(device)

# TODO 1-2: 5개 클래스에 대한 프롬프트를 정의해봅시다.
class_prompts = {
    "cat": "a photo of a cute domestic cat sitting, high quality, detailed",
    "dog": "a photo of a golden retriever dog sitting, high quality, detailed",
    "car": "a photo of a red sports car on a road, high quality, detailed",
    "airplane": "a photo of a commercial airplane flying in the sky, high quality, detailed",
    "flower": "a photo of a beautiful sunflower in a garden, high quality, detailed",
}

# TODO 1-3: 각 클래스별로 train 2장, test 1장의 이미지를 생성하여 저장해봅시다.
train_seeds = [42, 777]
test_seeds = [2024]

for split, seeds in [("train", train_seeds), ("test", test_seeds)]:
    for cls, prompt in class_prompts.items():
        os.makedirs(f"data/{split}/{cls}", exist_ok=True)
        for i, seed in enumerate(seeds):
            generator = torch.Generator(device=device).manual_seed(seed)
            result = pipe(
                prompt=prompt,
                guidance_scale=7.5,
                num_inference_steps=30,
                height=512,
                width=512,
                generator=generator,
            )
            img = result.images[0]
            img.save(f"data/{split}/{cls}/{cls}_{split}_{i}.png")
            print(f"[{split}] {cls}_{i} 저장 완료 (seed={seed})")

print(f"\n데이터 생성 완료! train: {len(train_seeds)}장/클래스, test: {len(test_seeds)}장/클래스")

In [ ]:
# TODO 2: 학습/테스트 transforms를 정의하고, ImageFolder와 DataLoader를 생성해봅시다.

# TODO 2-1: 학습용 transforms를 정의해봅시다. (RandomResizedCrop, RandomHorizontalFlip, ToTensor, Normalize)
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# TODO 2-2: 테스트용 transforms를 정의해봅시다. (Resize, ToTensor, Normalize)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# TODO 2-3: ImageFolder로 데이터셋을 생성해봅시다.
train_dataset = ImageFolder("data/train", transform=train_transforms)
test_dataset = ImageFolder("data/test", transform=test_transforms)

# TODO 2-4: DataLoader를 생성해봅시다. (batch_size=4)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print(f"학습 이미지: {len(train_dataset)}장, 테스트 이미지: {len(test_dataset)}장")
print(f"클래스: {train_dataset.classes}")
print(f"클래스→인덱스: {train_dataset.class_to_idx}")

💡ImageFolder 사용 팁
> `ImageFolder`는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">폴더명을 알파벳 순으로 정렬</mark>하여 클래스 인덱스를 자동 부여합니다.</br>
> 예: airplane=0, car=1, cat=2, dog=3, flower=4</br>
> 학습용 transforms에는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">RandomResizedCrop, RandomHorizontalFlip</mark> 등 데이터 증강을 적용하고, 테스트용에는 단순 Resize만 적용합니다.

</br>

## 모델 설정 (Freeze + fc 교체)

In [ ]:
# TODO 3: ResNet18 사전학습 모델을 로드하고, 모든 파라미터를 동결한 뒤, fc 레이어를 5개 클래스에 맞게 교체해봅시다.

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# 특성 추출기 동결
for param in model.parameters():
    param.requires_grad = False

# 분류 헤드 교체
num_classes = 5
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 파라미터: {trainable:,} (fc: {model.fc.in_features} → {num_classes})")

</br>

## 학습 설정

In [ ]:
# TODO 4: fc 레이어의 파라미터만 학습하도록 옵티마이저를 설정하고, 분류용 손실 함수와 학습 에폭 수를 정의해봅시다.

optimizer = optim.SGD(
    model.fc.parameters(),     # fc만 학습
    lr=0.01,
    momentum=0.9
)

criterion = nn.CrossEntropyLoss()
num_epochs = 3
print(f"옵티마이저: SGD (lr=0.01, momentum=0.9)")
print(f"손실 함수: CrossEntropyLoss")
print(f"에폭 수: {num_epochs}")

💡SGD vs Adam
> 전이학습에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">SGD + momentum</mark>은 안정적인 수렴을 보입니다.</br>
> 3~5 에폭만으로도 충분한 성능을 달성할 수 있습니다.

</br>

## 학습 루프

In [ ]:
# TODO 5: 에폭만큼 반복하며 학습 데이터로 모델을 학습시키고, 테스트 데이터로 정확도를 평가하여 에폭별 손실과 정확도를 출력해봅시다.

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # TODO 5-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad()
        # TODO 5-2: 순전파를 수행해봅시다.
        outputs = model(images)
        # TODO 5-3: 손실을 계산해봅시다.
        loss = criterion(outputs, labels)
        # TODO 5-4: 역전파를 수행해봅시다.
        loss.backward()
        # TODO 5-5: 파라미터를 업데이트해봅시다.
        optimizer.step()
        running_loss += loss.item()

    # 검증
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, Test Acc={correct/total:.2%}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">Epoch</th>
      <th style="text-align:center">Loss</th>
      <th style="text-align:center">Val Acc</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">1</td><td style="text-align:center">1.2345</td><td style="text-align:center">72.50%</td></tr>
    <tr><td style="text-align:center">2</td><td style="text-align:center">0.5678</td><td style="text-align:center">85.00%</td></tr>
    <tr><td style="text-align:center">3</td><td style="text-align:center">0.3456</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">90.00%</mark></td></tr>
  </tbody>
</table>

## 모델 평가 및 시각화

In [ ]:
# TODO 6: 역정규화 함수를 정의하고, 테스트 데이터에 대해 예측 결과를 시각화해봅시다.

# 역정규화 함수: 정규화된 텐서를 [0, 1] 범위로 복원
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

def denormalize(img_tensor, mean=mean, std=std):
    """정규화된 이미지 텐서를 [0, 1] 범위로 복원합니다."""
    mean_t = torch.tensor(mean).view(-1, 1, 1)
    std_t = torch.tensor(std).view(-1, 1, 1)
    img = img_tensor.cpu() * std_t + mean_t
    return img.clamp(0, 1)

# 테스트 데이터 평가 및 시각화
model.eval()
class_names = test_dataset.classes
correct, total = 0, 0
images_to_show, preds_to_show, labels_to_show = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        for img, pred, label in zip(images, predicted, labels):
            images_to_show.append(denormalize(img))
            preds_to_show.append(class_names[pred])
            labels_to_show.append(class_names[label])

print(f"테스트 정확도: {correct/total:.2%}")

# matplotlib으로 시각화
n = len(images_to_show)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1:
    axes = [axes]
for i, (img, pred, label) in enumerate(zip(images_to_show, preds_to_show, labels_to_show)):
    axes[i].imshow(img.permute(1, 2, 0).numpy())
    color = "green" if pred == label else "red"
    axes[i].set_title(f"예측: {pred}\n정답: {label}", color=color, fontsize=10)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

💡역정규화 (Denormalization)
> 모델에 입력하기 전 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 평균/표준편차로 정규화</mark>한 이미지는 사람이 볼 수 없는 값 범위를 갖습니다.</br>
> 시각화할 때는 `img * std + mean`으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">원래 [0, 1] 범위로 복원</mark>해야 올바른 색상이 표시됩니다.

</br>

# Knowledge Distillation (지식 증류)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">큰 모델(Teacher)의 지식을 작은 모델(Student)에 전달</mark>하는 모델 압축 기법입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구성요소</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Teacher</td><td>큰 사전학습 모델 (성능 높음)</td></tr>
    <tr><td style="text-align:center">Student</td><td>작은 모델 (경량, 배포용)</td></tr>
    <tr><td style="text-align:center">Soft Labels</td><td>Teacher의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">확률 분포</mark> (temperature 적용)</td></tr>
    <tr><td style="text-align:center">KD Loss</td><td>Soft Labels와 Student 출력 간 KL Divergence</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 7: Teacher 모델의 logits에 Temperature를 적용하여 Soft Labels을 계산하고, Hard Labels과 확률 분포가 어떻게 달라지는지 비교해봅시다.

# Temperature Scaling 예시
teacher_logits = torch.tensor([5.0, 2.0, 1.0])
T = 3.0  # Temperature

hard_labels = F.softmax(teacher_logits, dim=0)
soft_labels = F.softmax(teacher_logits / T, dim=0)

print(f"Hard Labels (T=1): {hard_labels.data.round(decimals=3)}")
print(f"Soft Labels (T=3): {soft_labels.data.round(decimals=3)}")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">클래스</th>
      <th style="text-align:center">Hard Label (T=1)</th>
      <th style="text-align:center">Soft Label (T=3)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Class 0</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.936</mark></td><td style="text-align:center">0.592</td></tr>
    <tr><td style="text-align:center">Class 1</td><td style="text-align:center">0.047</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.243</mark></td></tr>
    <tr><td style="text-align:center">Class 2</td><td style="text-align:center">0.017</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">0.165</mark></td></tr>
  </tbody>
</table>

💡Temperature의 역할
> Temperature > 1로 설정하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">확률 분포가 부드러워져</mark> 클래스 간 관계 정보가 전달됩니다.</br>
> 예: [0.9, 0.05, 0.05] → [0.6, 0.2, 0.2] (더 많은 정보 포함)

💡왜 KD를 사용하는가?
> Hard Label(정답 라벨)보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Soft Label이 더 풍부한 정보</mark>를 포함합니다.</br>
> "고양이와 호랑이는 비슷하다"는 관계 정보가 Soft Label에 암묵적으로 담깁니다.

</br>

## Knowledge Distillation 구현
> 앞에서 학습한 개념을 바탕으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Teacher-Student 구조의 Knowledge Distillation</mark>을 직접 구현합니다.</br></br>
> **KD Loss 공식:**</br>
> `KD_Loss = alpha * T² * KL_Divergence(student_soft, teacher_soft) + (1 - alpha) * CE_Loss(student_hard, true_label)`</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Teacher(ResNet-50)</mark>의 soft label을 활용하여 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Student(ResNet-18)</mark>를 학습시키고, 기본 학습(TODO 5)과 성능을 비교합니다.

In [ ]:
# TODO 8: ResNet-50 Teacher 모델을 로드하고, fc 레이어를 교체한 뒤 합성 데이터로 5 에폭 학습시켜봅시다.

# Teacher 모델 로드 및 동결
teacher_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
for param in teacher_model.parameters():
    param.requires_grad = False

# fc 레이어 교체 (5개 클래스)
teacher_model.fc = nn.Linear(teacher_model.fc.in_features, num_classes)
teacher_model = teacher_model.to(device)

# Teacher 학습 설정
teacher_optimizer = optim.SGD(teacher_model.fc.parameters(), lr=0.01, momentum=0.9)
teacher_criterion = nn.CrossEntropyLoss()
teacher_epochs = 5

# Teacher 학습 루프
for epoch in range(teacher_epochs):
    teacher_model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        teacher_optimizer.zero_grad()
        outputs = teacher_model(images)
        loss = teacher_criterion(outputs, labels)
        loss.backward()
        teacher_optimizer.step()
        running_loss += loss.item()

    print(f"Teacher Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}")

# Teacher 평가
teacher_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = teacher_model(images)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
print(f"\nTeacher 테스트 정확도: {correct/total:.2%}")

💡Teacher 모델 선택
> Teacher 모델은 Student보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">더 크고 성능이 높은 모델</mark>을 사용합니다.</br>
> ResNet-50(약 2,500만 파라미터)은 ResNet-18(약 1,100만 파라미터)보다 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">2배 이상 큰 모델</mark>이므로 Teacher로 적합합니다.

In [ ]:
# TODO 9: Knowledge Distillation 학습 루프를 구현해봅시다. (T=4.0, alpha=0.7, 10 에폭)

# KD 하이퍼파라미터
T = 4.0       # Temperature
alpha = 0.7   # KL Loss 비중

# Student 모델 새로 초기화 (ResNet-18)
student_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for param in student_model.parameters():
    param.requires_grad = False
student_model.fc = nn.Linear(student_model.fc.in_features, num_classes)
student_model = student_model.to(device)

student_optimizer = optim.SGD(student_model.fc.parameters(), lr=0.01, momentum=0.9)
ce_criterion = nn.CrossEntropyLoss()

teacher_model.eval()  # Teacher는 eval 모드 고정
student_model.train()
kd_epochs = 10

for epoch in range(kd_epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # TODO 9-1: Teacher의 soft labels을 생성해봅시다. (torch.no_grad, F.softmax with T)
        with torch.no_grad():
            teacher_logits = teacher_model(images)
            teacher_soft = F.softmax(teacher_logits / T, dim=1)

        # TODO 9-2: Student의 log-softmax를 Temperature T로 계산해봅시다.
        student_logits = student_model(images)
        student_log_soft = F.log_softmax(student_logits / T, dim=1)

        # TODO 9-3: KL Divergence 손실을 계산해봅시다. (T² 보정 포함)
        kl_loss = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean') * (T * T)

        # TODO 9-4: Cross-Entropy 손실을 계산해봅시다.
        ce_loss = ce_criterion(student_logits, labels)

        # TODO 9-5: 전체 KD 손실을 계산해봅시다. (alpha 가중합)
        loss = alpha * kl_loss + (1 - alpha) * ce_loss

        # TODO 9-6: 역전파 및 파라미터 업데이트를 수행해봅시다.
        student_optimizer.zero_grad()
        loss.backward()
        student_optimizer.step()

        running_loss += loss.item()

    print(f"KD Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}")

In [ ]:
# TODO 10: KD Student 모델을 테스트 데이터로 평가하고, TODO 5의 기본 학습 결과와 비교해봅시다.

student_model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = student_model(images)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

kd_accuracy = correct / total
print(f"KD Student 테스트 정확도: {kd_accuracy:.2%}")
print(f"\n--- 결과 비교 ---")
print(f"TODO 5 기본 ResNet-18 (CE Loss만 학습): 위 학습 결과 참고")
print(f"TODO 9 KD Student ResNet-18 (Teacher soft label 활용): {kd_accuracy:.2%}")
print(f"\nKnowledge Distillation을 통해 Teacher의 지식이 Student에게 전달되었습니다.")

💡KD 효과 비교
> Knowledge Distillation의 핵심은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Hard Label(0 또는 1)보다 Soft Label이 더 풍부한 학습 신호를 제공</mark>한다는 점입니다.</br>
> 예를 들어 "고양이" 이미지에 대해 Teacher가 [cat: 0.7, dog: 0.2, ...]로 예측했다면, "고양이와 개가 시각적으로 유사하다"는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 간 관계 정보</mark>가 Student에게 전달됩니다.</br>
> 소량의 합성 데이터에서 KD의 효과가 크게 나타날 수 있으며, 데이터가 충분할 경우에는 차이가 줄어들 수 있습니다.